# Debugging with `pdb` in a notebook

`layer2` is declared with the **wrong in_features** on purpose (`25` instead of `20`). Uncomment `pdb.set_trace()` (or `breakpoint()`) to inspect `x.shape` before the crash.

In the pdb prompt:

| Command | Meaning |
| --- | --- |
| `p x.shape` | print the tensor shape |
| `p self.layer2` | inspect the layer |
| `n` | next line |
| `c` | continue |
| `q` | quit |

Fix: `nn.Linear(20, 6)`.


In [ ]:
import torch
import torch.nn as nn


class BrokenNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(10, 20)
        self.activation = nn.ReLU()
        self.layer2 = nn.Linear(25, 6)  # bug: 25 should be 20

    def forward(self, x):
        print("initial shape:", tuple(x.shape))
        x = self.layer1(x)
        print("after layer 1:", tuple(x.shape))
        x = self.activation(x)
        print("after activation:", tuple(x.shape))
        # pdb.set_trace()  # uncomment to inspect x before the error
        x = self.layer2(x)
        return x


net = BrokenNet()
try:
    net(torch.randn(32, 10))
except RuntimeError as e:
    print("RuntimeError (expected):")
    print(e)


## Corrected version


In [ ]:
class FixedNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(10, 20)
        self.activation = nn.ReLU()
        self.layer2 = nn.Linear(20, 6)

    def forward(self, x):
        x = self.activation(self.layer1(x))
        return self.layer2(x)


print("fixed output:", FixedNet()(torch.randn(32, 10)).shape)
